# 11a — Batch Normalization

Corresponds to: Note 11 — Batch Normalization

**Libraries introduced in this notebook:** No new libraries — numpy only, introduced in Note 03.

**What this notebook demonstrates:**
1. Batch norm forward pass — four steps implemented from scratch, worked example verified against Note 11 Part 2
2. Covariate shift — activation distributions across layers with and without batch norm


In [ ]:
import numpy as np

## Part 1: Batch Norm Forward Pass

Batch normalization applies four steps to a mini-batch of pre-activations z:

    Step 1 — batch mean:      μ_B  = (1/m) Σ zᵢ
    Step 2 — batch variance:  σ²_B = (1/m) Σ (zᵢ − μ_B)²
    Step 3 — normalize:       ẑᵢ   = (zᵢ − μ_B) / √(σ²_B + ε)
    Step 4 — scale and shift: yᵢ   = γ · ẑᵢ + β

We reproduce the worked example from Note 11 Part 2: z = [2, 4, 6, 8], γ = 2.0, β = 1.0.

In [ ]:
z     = np.array([2.0, 4.0, 6.0, 8.0])
gamma = 2.0
beta  = 1.0
eps   = 0.0

# Step 1 — batch mean
mu_B = z.mean()
print(f'mu_B  = {mu_B}')          # expected: 5.0

# Step 2 — batch variance
var_B = z.var()
print(f'var_B = {var_B}')         # expected: 5.0

# Step 3 — normalize
z_hat = (z - mu_B) / np.sqrt(var_B + eps)
print(f'z_hat = {np.round(z_hat, 3)}')  # expected: [-1.342, -0.447, 0.447, 1.342]

# Step 4 — scale and shift
y = gamma * z_hat + beta
print(f'y     = {np.round(y, 3)}')       # expected: [-1.684, 0.106, 1.894, 3.684]


The outputs match the hand calculation in Note 11 Part 2 exactly.

Now wrap the four steps into a reusable function that handles a batch of vectors (shape: m × features).

In [ ]:
def batch_norm_forward(z, gamma, beta, eps=1e-8):
    mu_B  = z.mean(axis=0)
    var_B = z.var(axis=0)
    z_hat = (z - mu_B) / np.sqrt(var_B + eps)
    y     = gamma * z_hat + beta
    return y, z_hat, mu_B, var_B

# Verify on the worked example
z_batch = np.array([[2.0], [4.0], [6.0], [8.0]])  # shape (4, 1)
y_out, _, mu, var = batch_norm_forward(z_batch, gamma=2.0, beta=1.0, eps=0.0)
print(f'y = {np.round(y_out.flatten(), 3)}')
print(f'output mean = {y_out.mean():.3f}   expected: 1.0')
print(f'output std  = {y_out.std():.3f}    expected: 2.0')


## Part 2: Covariate Shift Demonstration

Internal covariate shift means activation distributions drift across layers during training.
We simulate a 4-layer network and compare activation statistics with and without batch norm.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def forward_pass(x, weights, use_bn):
    """Forward pass through a list of weight matrices. Returns per-layer activation stats."""
    a = x
    stats = []
    for W in weights:
        z = a @ W
        if use_bn:
            z, _, _, _ = batch_norm_forward(z, gamma=1.0, beta=0.0)
        a = relu(z)
        stats.append((a.mean(), a.std()))
    return stats

rng = np.random.default_rng(0)
x = rng.standard_normal((100, 16))
weights = [rng.standard_normal((16, 16)) * 2.0 for _ in range(4)]

print('Without batch norm:')
for i, (mean, std) in enumerate(forward_pass(x, weights, use_bn=False)):
    print(f'  Layer {i+1}: mean={mean:.3f}  std={std:.3f}')

print('\nWith batch norm:')
for i, (mean, std) in enumerate(forward_pass(x, weights, use_bn=True)):
    print(f'  Layer {i+1}: mean={mean:.3f}  std={std:.3f}')


Without batch norm, mean and std drift across layers — each layer receives a different distribution.
With batch norm, mean and std remain consistent — each layer sees a stable input distribution.

## Summary

This notebook demonstrated:

1. **Batch norm forward pass** — four steps implemented explicitly. Worked example matches Note 11 Part 2.
2. **Covariate shift** — without batch norm, activation distributions drift across layers. With batch norm, they remain stable.
